# Phase 3 — Analyse, Visualise & Report
**Goal:** Tell the story hidden in the data through 5+ charts, groupby analysis, and all required math tasks (manual stats, z-score, cosine similarity, probability).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from numpy.linalg import norm

df = pd.read_csv('data/cleaned/Ames_Housing_Features.csv')
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')

---
# Part A — Exploratory Data Analysis (EDA)

## Chart 1 — Histogram / KDE: Distribution of SalePrice
**Question:** How are house prices distributed across the dataset?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['SalePrice'], kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Distribution of SalePrice')
axes[0].set_xlabel('Sale Price ($)')
axes[0].set_ylabel('Frequency')

sns.histplot(df['Gr Liv Area'], kde=True, color='darkorange', ax=axes[1])
axes[1].set_title('Distribution of Above-Ground Living Area (sq ft)')
axes[1].set_xlabel('Living Area (sq ft)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print("""
INSIGHT — Chart 1:
SalePrice is right-skewed: the majority of homes sell between $100,000 and $250,000,
but a long tail of high-value properties extends the distribution rightward. This skew
is why a log-transform was applied in Phase 2 — linear models assume normality and
would otherwise be dominated by those expensive outliers. Similarly, Gr Liv Area shows
a moderate right skew: most homes are 1,000–2,500 sq ft, with a few very large homes
creating the tail. Both distributions suggest that the 'typical' Ames buyer purchases
a mid-sized, mid-priced home.
""")

## Chart 2 — Distribution of Year Built
**Question:** When were most Ames houses constructed?

In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(df['Year Built'], bins=40, kde=True, color='mediumpurple')
plt.title('Distribution of Year Built')
plt.xlabel('Year Built')
plt.ylabel('Number of Houses')
plt.tight_layout()
plt.show()

print("""
INSIGHT — Chart 2:
The Year Built distribution shows two clear construction booms: one in the post-WWII
era (1950s–1960s) and a much larger boom from the 1990s through 2008. The sharp drop
after 2008 reflects the U.S. housing market crash and subsequent construction slowdown.
This bimodal pattern means Ames has a mix of older and newer housing stock, and the
House_Age_Category feature we created in Phase 2 captures this division meaningfully.
""")

## Chart 3 — Grouped Boxplots: SalePrice by Overall Quality
**Question:** Does higher quality consistently translate to a higher sale price?

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='Overall Qual', y='SalePrice', data=df, palette='Blues')
plt.title('SalePrice by Overall Quality Rating (1=Poor, 10=Excellent)')
plt.xlabel('Overall Quality')
plt.ylabel('Sale Price ($)')
plt.tight_layout()
plt.show()

print("""
INSIGHT — Chart 3:
There is a strong, monotonically increasing relationship between Overall Quality and
SalePrice. Houses rated 9–10 sell for roughly 3–4x the price of houses rated 3–4.
The interquartile range also widens at higher quality levels, meaning premium homes
have more price variability — likely because other factors (neighbourhood, lot size)
further differentiate them. This confirms that Overall Qual is one of the most
powerful predictors in the dataset.
""")

## Chart 4 — Grouped Boxplots: SalePrice by House Age Category
**Question:** Do newer houses command higher prices than older ones?

In [ ]:
plt.figure(figsize=(8, 6))
order = ['Old', 'Recent', 'New']
sns.boxplot(x='House_Age_Category', y='SalePrice', data=df, order=order, palette='Set2')
plt.title('SalePrice by House Age Category')
plt.xlabel('Age Category (Old: pre-1950, Recent: 1950–2000, New: 2000+)')
plt.ylabel('Sale Price ($)')
plt.tight_layout()
plt.show()

print("""
INSIGHT — Chart 4:
Newer houses ('New', built after 2000) have a notably higher median sale price and
a wider spread than older categories. 'Old' houses (pre-1950) have the lowest median
but a surprisingly wide upper range — some older homes in desirable neighbourhoods
or with renovations still command high prices. This non-linear relationship between
age and price is why raw Year Built alone is less informative than the binned
category we created.
""")

## Chart 5 — Correlation Heatmap: Top 10 Features vs SalePrice
**Question:** Which numeric features are most strongly correlated with sale price?

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
top_features = numeric_df.corr()['SalePrice'].abs().sort_values(ascending=False).head(10).index

plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df[top_features].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Top 10 Features Most Correlated with SalePrice')
plt.tight_layout()
plt.show()

print("""
INSIGHT — Chart 5:
The heatmap confirms Overall Qual (r≈0.80) and Gr Liv Area (r≈0.71) as the top two
individual predictors of SalePrice. Our engineered feature Qual_x_Area appears near
the top, validating the hypothesis that combining quality and space captures more
signal than either alone. Notably, Total Bsmt SF and 1st Flr SF are highly correlated
with each other (r≈0.82), confirming our Phase 2 decision to drop one via the
redundancy filter.
""")

## Chart 6 — Scatter Plot: Living Area vs SalePrice (coloured by Quality)
**Question:** Is the size–price relationship consistent across quality levels, or does quality moderate it?

In [ ]:
plt.figure(figsize=(12, 7))
scatter = plt.scatter(
    df['Gr Liv Area'],
    df['SalePrice'],
    c=df['Overall Qual'],
    cmap='RdYlGn',
    alpha=0.6,
    edgecolors='none',
    s=40
)
plt.colorbar(scatter, label='Overall Quality (1–10)')
plt.title('Living Area vs SalePrice — Coloured by Overall Quality')
plt.xlabel('Above-Ground Living Area (sq ft)')
plt.ylabel('Sale Price ($)')
plt.tight_layout()
plt.show()

print("""
INSIGHT — Chart 6:
The scatter plot reveals a clear positive linear relationship between living area and
sale price. More importantly, the colour coding shows that high-quality homes (green)
cluster at the top of the price range for any given size, while low-quality homes
(red) consistently sell below comparable-sized better-quality homes. This 'quality
premium' is visible as a vertical spread in the data — two houses of the same size
can differ by $100,000+ depending on quality. This justifies our Qual_x_Area
interaction feature, which captures exactly this combined effect.
""")

## Groupby Summary: Mean SalePrice by Neighbourhood
**Question:** Which neighbourhoods command the highest and lowest average prices?

In [ ]:
neighborhood_avg = df.groupby('Neighborhood')['SalePrice'].mean().sort_values(ascending=False)

plt.figure(figsize=(14, 6))
neighborhood_avg.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Mean SalePrice by Neighbourhood')
plt.xlabel('Neighbourhood')
plt.ylabel('Mean Sale Price ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('Top 5 most expensive neighbourhoods:')
print(neighborhood_avg.head().apply(lambda x: f'${x:,.0f}'))
print('\nTop 5 most affordable neighbourhoods:')
print(neighborhood_avg.tail().apply(lambda x: f'${x:,.0f}'))

print("""
INSIGHT — Groupby:
There is a dramatic spread in average prices across neighbourhoods — the most
expensive neighbourhood has a mean price more than 3x that of the cheapest.
This confirms that location is a critical driver of value and that adding
Neighbourhood as a feature (or neighbourhood-level average price) would
meaningfully improve a predictive model.
""")

---
# Part B — Math Basics

## Math Task 1 — Manual Mean and Standard Deviation
We compute the mean and standard deviation of SalePrice by hand using NumPy, then verify against pandas.

In [ ]:
price_array = df['SalePrice'].to_numpy()

# Manual mean: sum of all values divided by count
manual_mean = np.sum(price_array) / len(price_array)

# Manual std: square root of the average squared deviation from the mean
manual_std = np.sqrt(np.sum((price_array - manual_mean) ** 2) / len(price_array))

print('=== Manual vs Pandas Comparison ===')
print(f'Manual Mean:  ${manual_mean:,.2f}')
print(f'Pandas Mean:  ${df["SalePrice"].mean():,.2f}')
print(f'Match: {np.isclose(manual_mean, df["SalePrice"].mean())}\n')

print(f'Manual Std:   ${manual_std:,.2f}')
# pandas uses ddof=1 (sample std) by default; we use population std — use ddof=0 to match
print(f'Pandas Std (population, ddof=0): ${df["SalePrice"].std(ddof=0):,.2f}')
print(f'Match: {np.isclose(manual_std, df["SalePrice"].std(ddof=0))}')

## Math Task 2 — Manual Standardisation (Z-Score) vs StandardScaler
We standardise `SalePrice` by hand using broadcasting, then compare to sklearn's `StandardScaler` output to verify they match.

In [ ]:
# Manual z-score: subtract the mean, divide by standard deviation
z_manual = (price_array - manual_mean) / manual_std

# StandardScaler z-score (uses population std internally)
scaler = StandardScaler()
z_sklearn = scaler.fit_transform(price_array.reshape(-1, 1)).flatten()

print('=== Manual Z-Score vs StandardScaler ===')
print(f'Manual z-score   (first 5): {z_manual[:5].round(4)}')
print(f'StandardScaler   (first 5): {z_sklearn[:5].round(4)}')
print(f'All values match: {np.allclose(z_manual, z_sklearn)}')
print(f'\nManual z-score — mean: {z_manual.mean():.6f} (expected ~0)')
print(f'Manual z-score — std:  {z_manual.std():.6f}  (expected ~1)')

## Math Task 3 — Cosine Similarity
Cosine similarity measures the angle between two feature vectors. A value of 1 means identical direction; 0 means orthogonal. We compare the feature vectors of the most expensive and least expensive houses.

In [ ]:
# Select only numeric columns for the feature vectors
numeric_df = df.select_dtypes(include=[np.number])

# Identify the highest-value and lowest-value records
idx_high = df['SalePrice'].idxmax()
idx_low  = df['SalePrice'].idxmin()

vec_high = numeric_df.loc[idx_high].fillna(0).values.astype(float)
vec_low  = numeric_df.loc[idx_low].fillna(0).values.astype(float)

# Cosine similarity = dot(A, B) / (||A|| * ||B||)
cos_sim = np.dot(vec_high, vec_low) / (norm(vec_high) * norm(vec_low))

print('=== Cosine Similarity: Most Expensive vs Least Expensive House ===')
print(f'Most expensive house SalePrice: ${df.loc[idx_high, "SalePrice"]:,.0f}')
print(f'Least expensive house SalePrice: ${df.loc[idx_low, "SalePrice"]:,.0f}')
print(f'Cosine Similarity: {cos_sim:.4f}')
print("""
Interpretation: A cosine similarity close to 1 means both houses have similar
feature proportions (same 'direction' in feature space) despite very different
absolute values. A lower value would indicate the two houses are structurally
quite different in terms of their numeric characteristics.
""")

## Math Task 4 — Probability Estimate
We estimate the probability that a high-quality house (Overall Qual ≥ 8) sells above the 75th percentile price threshold.

In [ ]:
price_threshold = df['SalePrice'].quantile(0.75)
high_quality_houses = df[df['Overall Qual'] >= 8]

prob = (high_quality_houses['SalePrice'] > price_threshold).mean()

# Baseline: probability for ALL houses
baseline_prob = (df['SalePrice'] > price_threshold).mean()

print('=== Probability Estimate ===')
print(f'75th percentile price threshold: ${price_threshold:,.0f}')
print(f'Number of high-quality houses (Qual >= 8): {len(high_quality_houses)}')
print(f'\nP(SalePrice > 75th pct | Overall Qual >= 8): {prob:.2%}')
print(f'Baseline P(SalePrice > 75th pct | all houses): {baseline_prob:.2%}')
print(f'\nHigh-quality homes are {prob/baseline_prob:.1f}x more likely to sell above the 75th percentile')
print("""
Interpretation: This probability confirms what the correlation analysis suggested —
quality is one of the strongest drivers of price. A house rated 8+ is dramatically
more likely to sell in the top quarter of the market than an average house.
""")

---
# Bonus — Dashboard: 4 Charts in One Figure

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Ames Housing — Key Insights Dashboard', fontsize=16, fontweight='bold', y=1.01)

# Panel 1: SalePrice distribution
sns.histplot(df['SalePrice'], kde=True, color='steelblue', ax=axes[0, 0])
axes[0, 0].set_title('SalePrice Distribution')
axes[0, 0].set_xlabel('Sale Price ($)')
axes[0, 0].set_ylabel('Count')

# Panel 2: Quality vs Price boxplot
sns.boxplot(x='Overall Qual', y='SalePrice', data=df, palette='Blues', ax=axes[0, 1])
axes[0, 1].set_title('SalePrice by Overall Quality')
axes[0, 1].set_xlabel('Overall Quality')
axes[0, 1].set_ylabel('Sale Price ($)')

# Panel 3: Scatter — Area vs Price coloured by Quality
sc = axes[1, 0].scatter(df['Gr Liv Area'], df['SalePrice'],
                         c=df['Overall Qual'], cmap='RdYlGn', alpha=0.5, s=15)
fig.colorbar(sc, ax=axes[1, 0], label='Quality')
axes[1, 0].set_title('Living Area vs SalePrice (colour = Quality)')
axes[1, 0].set_xlabel('Gr Liv Area (sq ft)')
axes[1, 0].set_ylabel('Sale Price ($)')

# Panel 4: Top 8 neighbourhood mean prices
top_neighborhoods = neighborhood_avg.head(8)
top_neighborhoods.plot(kind='bar', ax=axes[1, 1], color='mediumpurple', edgecolor='white')
axes[1, 1].set_title('Top 8 Neighbourhoods by Mean SalePrice')
axes[1, 1].set_xlabel('Neighbourhood')
axes[1, 1].set_ylabel('Mean Sale Price ($)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
print('Dashboard saved!')